# 🥉 → 🥈 Medications : Bronze to Silver Transformation



| # | Source Column | Target Column | Transformation Type |
|---|---|---|---|
| 1  | medication_id                | medication_id          | Deduplication (PK)             |
| 2  | drug_name                    | drug_name_clean        | initcap + TRIM; NULL → brand   |
| 3  | brand_name                   | brand_name_clean       | Remove ®/™ symbols + TRIM      |
| 4  | drug_name + brand_name       | drug_display_name      | Derived best display name      |
| 5  | rxnorm_code                  | rxnorm_code_clean      | Cast STRING; reject non-numeric|
| 6  | ndc_code                     | ndc_code_clean         | Pattern validation (NDC format)|
| 7  | dose_amount                  | dose_amount_valid      | Range validation (> 0 only)    |
| 8  | dose_unit                    | dose_unit_std          | UPPER + TRIM                   |
| 9  | route                        | route_std              | initcap + TRIM; NULL → Unknown |
| 10 | frequency                    | frequency_std          | Sig code expansion             |
| 11 | start_date                   | start_date_std         | Multi-format date parse        |
| 12 | end_date                     | end_date_std           | Multi-format date parse        |
| 13 | start_date + end_date        | actual_days_supply     | Derived datediff               |
| 14 | days_supply                  | days_supply_valid      | Range validation (> 0 only)    |
| 15 | medication_status            | medication_status_std  | initcap + TRIM; NULL → Unknown |
| 16 | indication_icd10             | indication_icd10_std   | UPPER + TRIM                   |
| 17 | dispensing_pharmacy          | pharmacy_std           | initcap + TRIM                 |
| 18 | prescriber_id                | prescriber_id_valid    | TRIM + FK reference note       |
| 19 | (new)                        | dq_med_flag            | Row-level DQ flag              |

## 1. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, DateType

## 2. Read Bronze Data

In [0]:
CSV_PATH = r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.medications.csv'

df_bronze_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CSV_PATH)
)

print(f"[BRONZE]  Row count : {df_bronze_raw.count()}")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(10))

## 3. Deduplication — `medication_id` (Primary Key)

In [0]:
df_deduped = df_bronze_raw.dropDuplicates(["medication_id"])

print(f"[DEDUP]   Before : {df_bronze_raw.count()}  |  After : {df_deduped.count()}")
display(df_deduped.limit(10))

## 4. Text Cleansing — `drug_name` → `drug_name_clean`
- initcap + TRIM
- NULL drug_name → fall back to brand_name (after symbol strip) as interim; final coalesce happens in Step 6

In [0]:
df_with_drug_name_clean = df_deduped.withColumn(
    "drug_name_clean",
    F.when(
        F.col("drug_name").isNotNull(),
        F.initcap(F.trim(F.col("drug_name")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_drug_name_clean.limit(10))

## 5. Text Cleansing — `brand_name` → `brand_name_clean`
- Strip trademark symbols  **®**  and  **™**  using `regexp_replace`
- TRIM whitespace
- NULL stays NULL

In [0]:
df_with_brand_name_clean = df_with_drug_name_clean.withColumn(
    "brand_name_clean",
    F.when(
        F.col("brand_name").isNotNull(),
        F.trim(
            F.regexp_replace(F.col("brand_name"), "[®™]", "")
        )
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_brand_name_clean.limit(10))

## 6. Derived — Best Display Name : `drug_display_name`
- `F.coalesce(drug_name_clean, brand_name_clean)`
- NULL only when BOTH source columns are NULL

In [0]:
df_with_drug_display_name = df_with_brand_name_clean.withColumn(
    "drug_display_name",
    F.coalesce(
        F.col("drug_name_clean"),
        F.col("brand_name_clean")
    )
)
#display(df_with_drug_display_name.limit(10))

## 7. Type Validation — `rxnorm_code` → `rxnorm_code_clean`
- Cast float source to STRING, strip decimal part
- Reject any value that is not purely numeric after cleaning → NULL
- RxNorm codes are positive integers (e.g. 526772)

In [0]:
df_with_rxnorm_clean = df_with_drug_display_name.withColumn(
    "rxnorm_code_clean",
    F.when(
        F.col("rxnorm_code").isNotNull(),
        F.when(
            F.regexp_extract(
                F.regexp_replace(
                    F.col("rxnorm_code").cast(StringType()),
                    "\\.0$", ""
                ),
                "^[0-9]+$", 0
            ) != "",
            F.regexp_replace(
                F.col("rxnorm_code").cast(StringType()),
                "\\.0$", ""
            )
        ).otherwise(F.lit(None).cast(StringType()))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_rxnorm_clean.limit(10))

## 8. Pattern Validation — `ndc_code` → `ndc_code_clean`
- Accepted NDC formats:
  - 10-digit hyphenated : `NNNNN-NNNN-NN`
  - 11-digit contiguous : `NNNNNNNNNNN`
- Non-matching values → NULL

In [0]:
df_with_ndc_clean = df_with_rxnorm_clean.withColumn(
    "ndc_code_clean",
    F.when(
        F.col("ndc_code").isNotNull(),
        F.when(
            F.regexp_extract(
                F.trim(F.col("ndc_code")),
                r"^(\d{5}-\d{4}-\d{2}|\d{11})$",
                0
            ) != "",
            F.trim(F.col("ndc_code"))
        ).otherwise(F.lit(None).cast(StringType()))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_ndc_clean.limit(10))

## 9. Range Validation — `dose_amount` → `dose_amount_valid`
- Cast to Double
- Keep only values **> 0**
- NULL or ≤ 0 → NULL

In [0]:
df_with_dose_amount_valid = df_with_ndc_clean.withColumn(
    "dose_amount_valid",
    F.when(
        F.col("dose_amount").cast(DoubleType()) > F.lit(0.0),
        F.col("dose_amount").cast(DoubleType())
    ).otherwise(F.lit(None).cast(DoubleType()))
)
#display(df_with_dose_amount_valid.limit(10))

## 10. Standardization — `dose_unit` → `dose_unit_std`
- UPPER + TRIM → MG / ML / MCG / IU / G / UNITS

In [0]:
df_with_dose_unit_std = df_with_dose_amount_valid.withColumn(
    "dose_unit_std",
    F.when(
        F.col("dose_unit").isNotNull(),
        F.upper(F.trim(F.col("dose_unit")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_dose_unit_std.limit(10))

## 11. Standardization — `route` → `route_std`
- initcap + TRIM
- Normalize aliases: `PO` and `ORAL` and `oral` → `Oral`; `SC` stays as `Sc` (acceptable); `IV` stays `Iv`
- NULL → `Unknown`

In [0]:
df_with_route_std = df_with_dose_unit_std.withColumn(
    "route_std",
    F.when(
        F.col("route").isNull(),
        F.lit("Unknown")
    ).when(
        F.upper(F.trim(F.col("route"))).isin("PO", "ORAL"),
        F.lit("Oral")
    ).when(
        F.upper(F.trim(F.col("route"))) == F.lit("IV"),
        F.lit("IV")
    ).when(
        F.upper(F.trim(F.col("route"))) == F.lit("IM"),
        F.lit("IM")
    ).when(
        F.upper(F.trim(F.col("route"))) == F.lit("SC"),
        F.lit("SC")
    ).otherwise(
        F.initcap(F.trim(F.col("route")))
    )
)
#display(df_with_route_std.limit(10))

## 12. Standardization — `frequency` → `frequency_std`
Sig code expansion (as per mapping sheet):

| Source values | Standardized output |
|---|---|
| OD / QD / od / Daily / Once | Once Daily |
| BD / BID / Twice daily | Twice Daily |
| TDS / TID | Three Times Daily |
| QID | Four Times Daily |
| SOS | As Needed |
| NULL | Unknown |

In [0]:
df_with_frequency_std = df_with_route_std.withColumn(
    "frequency_std",
    F.when(
        F.col("frequency").isNull(),
        F.lit("Unknown")
    ).when(
        F.upper(F.trim(F.col("frequency"))).isin("OD", "QD", "DAILY", "ONCE"),
        F.lit("Once Daily")
    ).when(
        F.upper(F.trim(F.col("frequency"))).isin("BD", "BID", "TWICE DAILY"),
        F.lit("Twice Daily")
    ).when(
        F.upper(F.trim(F.col("frequency"))).isin("TDS", "TID"),
        F.lit("Three Times Daily")
    ).when(
        F.upper(F.trim(F.col("frequency"))) == F.lit("QID"),
        F.lit("Four Times Daily")
    ).when(
        F.upper(F.trim(F.col("frequency"))) == F.lit("SOS"),
        F.lit("As Needed")
    ).otherwise(
        F.initcap(F.trim(F.col("frequency")))
    )
)
#display(df_with_frequency_std.limit(10))

## 13. Date Standardization — `start_date` → `start_date_std`

**Formats observed in data :**
- `MM-dd-yy`      → 03-07-21 / 11-03-20
- `dd/MM/yyyy`    → 23/06/2023 / 24/12/2019
- `MM-dd-yyyy`    → 12-10-2019 (edge)
- `dd-MM-yy`      → ambiguous; resolved by position in coalesce

`F.coalesce` tries each format in order; first non-null parse wins.
Unparseable values → NULL.

In [0]:
df_with_start_date_std = df_with_frequency_std.withColumn(
    "start_date_std",
    F.coalesce(
        F.expr("try_to_date(start_date, 'MM-dd-yy')"),
        F.expr("try_to_date(start_date, 'dd/MM/yyyy')"),
        F.expr("try_to_date(start_date, 'MM-dd-yyyy')"),
        F.expr("try_to_date(start_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(start_date, 'yyyy-MM-dd')"),
        F.expr("try_to_date(start_date, 'dd-MM-yy')")
    )
)
#display(df_with_start_date_std.limit(10))

## 14. Date Standardization — `end_date` → `end_date_std`
Same multi-format logic as start_date.
NULL end_date = ongoing / PRN medication — preserved as NULL.

In [0]:
df_with_end_date_std = df_with_start_date_std.withColumn(
    "end_date_std",
    F.coalesce(
        F.expr("try_to_date(end_date, 'MM-dd-yy')"),
        F.expr("try_to_date(end_date, 'dd/MM/yyyy')"),
        F.expr("try_to_date(end_date, 'MM-dd-yyyy')"),
        F.expr("try_to_date(end_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(end_date, 'yyyy-MM-dd')"),
        F.expr("try_to_date(end_date, 'dd-MM-yy')")
    )
)
#display(df_with_frequency_std.limit(10))

## 15. Derived — Actual Days Supply : `actual_days_supply`
- `F.datediff(end_date_std, start_date_std)`
- NULL when either date is NULL

In [0]:
df_with_actual_days_supply = df_with_end_date_std.withColumn(
    "actual_days_supply",
    F.when(
        F.col("start_date_std").isNotNull() & F.col("end_date_std").isNotNull(),
        F.datediff(F.col("end_date_std"), F.col("start_date_std"))
    ).otherwise(F.lit(None).cast(IntegerType()))
)
#display(df_with_actual_days_supply.limit(10))

## 16. Range Validation — `days_supply` → `days_supply_valid`
- Cast to Integer
- Keep only values **> 0**
- ≤ 0 or NULL → NULL (source has -1 sentinel values which are invalid)

In [0]:
df_with_days_supply_valid = df_with_actual_days_supply.withColumn(
    "days_supply_valid",
    F.when(
        F.col("days_supply").cast(IntegerType()) > F.lit(0),
        F.col("days_supply").cast(IntegerType())
    ).otherwise(F.lit(None).cast(IntegerType()))
)
#display(df_with_days_supply_valid.limit(10))

## 17. Standardization — `medication_status` → `medication_status_std`
- initcap + TRIM
- Normalizes: active / ACTIVE / Active → Active; discontinued / Discontinued → Discontinued
- NULL → `Unknown`

In [0]:
df_with_medication_status_std = df_with_days_supply_valid.withColumn(
    "medication_status_std",
    F.when(
        F.col("medication_status").isNull(),
        F.lit("Unknown")
    ).otherwise(
        F.initcap(F.trim(F.col("medication_status")))
    )
)
#display(df_with_medication_status_std.limit(10))

## 18. Standardization — `indication_icd10` → `indication_icd10_std`
- UPPER + TRIM
- Aligns with diagnoses silver table format (e.g. I10, K80.20, F32.9)

In [0]:
df_with_indication_icd10_std = df_with_medication_status_std.withColumn(
    "indication_icd10_std",
    F.when(
        F.col("indication_icd10").isNotNull(),
        F.upper(F.trim(F.col("indication_icd10")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_indication_icd10_std.limit(10))

## 19. Casing Standardization — `dispensing_pharmacy` → `pharmacy_std`
- initcap + TRIM (e.g. MedPlus → Medplus, Wellness Forever → Wellness Forever)

In [0]:
df_with_pharmacy_std = df_with_indication_icd10_std.withColumn(
    "pharmacy_std",
    F.when(
        F.col("dispensing_pharmacy").isNotNull(),
        F.initcap(F.trim(F.col("dispensing_pharmacy")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_pharmacy_std.limit(10))

## 20. FK Validation — `prescriber_id` → `prescriber_id_valid`
- TRIM the value
- NULL stays NULL
- NOTE: Full referential integrity check against `providers_silver` table
  should be run as a separate DQ job. This step only cleans the key.

In [0]:
df_with_prescriber_id_valid = df_with_pharmacy_std.withColumn(
    "prescriber_id_valid",
    F.when(
        F.col("prescriber_id").isNotNull(),
        F.trim(F.col("prescriber_id"))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_prescriber_id_valid.limit(10))

## 21. Data Quality Flag — `dq_med_flag`

**Flag = 1** when ANY of the following conditions are true :
- `drug_display_name` is NULL  (no usable drug name at all)
- `dose_amount_valid` is NULL  (missing or invalid dose)
- `start_date_std` is NULL     (missing or unparseable start date)

**Default = 0** (row passes all DQ checks)

In [0]:
df_with_dq_flag = df_with_prescriber_id_valid.withColumn(
    "dq_med_flag",
    F.when(
        F.col("drug_display_name").isNull()
        | F.col("dose_amount_valid").isNull()
        | F.col("start_date_std").isNull(),
        F.lit(1).cast(IntegerType())
    ).otherwise(F.lit(0).cast(IntegerType()))
)
#display(df_with_dq_flag.limit(10))

## 22. Select Final Silver Columns

In [0]:
df_silver_medications = df_with_dq_flag.select(

    # ── Identifiers (passthrough) ──────────────────────────────────────────
    F.col("medication_id"),
    F.col("encounter_id"),
    F.col("patient_id"),
    F.col("prescriber_id_valid"),
    F.col("ingestion_timestamp"),

    # ── Cleaned drug / brand name columns ─────────────────────────────────
    F.col("drug_name_clean"),
    F.col("brand_name_clean"),
    F.col("drug_display_name"),

    # ── Validated code columns ─────────────────────────────────────────────
    F.col("rxnorm_code_clean"),
    F.col("ndc_code_clean"),

    # ── Validated and standardized dose ────────────────────────────────────
    F.col("dose_amount_valid"),
    F.col("dose_unit_std"),

    # ── Standardized route and frequency ──────────────────────────────────
    F.col("route_std"),
    F.col("frequency_std"),

    # ── Standardized date columns ──────────────────────────────────────────
    F.col("start_date_std"),
    F.col("end_date_std"),

    # ── Derived supply / duration columns ─────────────────────────────────
    F.col("actual_days_supply"),
    F.col("days_supply_valid"),

    # ── Standardized status and clinical columns ───────────────────────────
    F.col("medication_status_std"),
    F.col("indication_icd10_std"),

    # ── Standardized pharmacy ──────────────────────────────────────────────
    F.col("pharmacy_std"),

    # ── Data Quality flag ──────────────────────────────────────────────────
    F.col("dq_med_flag")
)

print(f"[SILVER]  Final row count : {df_silver_medications.count()}")
df_silver_medications.printSchema()
display(df_silver_medications.limit(10))

## 24. Preview Silver Data

In [0]:
display(df_silver_medications.limit(20))

## 25. Write to Silver Layer (Delta)

In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "medications"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
df_silver_medications.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")

## 26. Verify Silver Table (post-write)

In [0]:
# Uncomment after running the write step above

# df_silver_verify = spark.read.format("delta").load(SILVER_DELTA_PATH)
# print(f"[VERIFY]  Silver row count : {df_silver_verify.count()}")
# display(df_silver_verify.limit(10))

---
## DataFrame Lineage (Transformation Chain)

```
df_bronze_raw
  └─► df_deduped
        └─► df_with_drug_name_clean
              └─► df_with_brand_name_clean
                    └─► df_with_drug_display_name
                          └─► df_with_rxnorm_clean
                                └─► df_with_ndc_clean
                                      └─► df_with_dose_amount_valid
                                            └─► df_with_dose_unit_std
                                                  └─► df_with_route_std
                                                        └─► df_with_frequency_std
                                                              └─► df_with_start_date_std
                                                                    └─► df_with_end_date_std
                                                                          └─► df_with_actual_days_supply
                                                                                └─► df_with_days_supply_valid
                                                                                      └─► df_with_medication_status_std
                                                                                            └─► df_with_indication_icd10_std
                                                                                                  └─► df_with_pharmacy_std
                                                                                                        └─► df_with_prescriber_id_valid
                                                                                                              └─► df_with_dq_flag
                                                                                                                    └─► df_silver_medications ✅
```